# Keystone Eval Analysis — 2026-02-25

Comparing: **claude-haiku**, **claude-opus**, **codex-gpt-5.2**, **codex-mini-gpt-5.1**

---

> **Note:** This notebook is checked in without cell outputs.  
> To execute and save results to a separate file, run:
> ```bash
> uv run jupyter nbconvert --to notebook --execute evals/eda/keystone_eval_analysis.ipynb --output keystone_eval_analysis_executed.ipynb
> ```

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

BASE = Path.home() / "keystone_evals" / "2026-02-25_thad"

records = []
for p in BASE.rglob("eval_result.json"):
    with p.open() as f:
        d = json.load(f)
    rel = p.relative_to(BASE)
    model = rel.parts[0]
    repo = rel.parts[1]
    trial = int(rel.parts[2].replace("trial_", ""))
    br = d.get("bootstrap_result") or {}
    ver = br.get("verification") or {}
    agent = br.get("agent") or {}
    cost = agent.get("cost") or {}
    repo_entry = d.get("repo_entry") or {}
    records.append(
        {
            "model": model,
            "repo": repo,
            "trial": trial,
            "success": d.get("success", False),
            "bootstrap_success": br.get("success", False),
            "verification_success": ver.get("success", False),
            "error_message": br.get("error_message", ""),
            "image_build_seconds": ver.get("image_build_seconds"),
            "test_execution_seconds": ver.get("test_execution_seconds"),
            "agent_duration_seconds": agent.get("duration_seconds"),
            "agent_timed_out": agent.get("timed_out", False),
            "cost_usd": cost.get("cost_usd"),
            "input_tokens": cost.get("token_spending", {}).get("input"),
            "output_tokens": cost.get("token_spending", {}).get("output"),
            "language": repo_entry.get("language", ""),
            "difficulty": repo_entry.get("difficulty", ""),
        }
    )

df = pd.DataFrame(records)
print(
    f"Loaded {len(df)} eval results: {df['model'].nunique()} models x {df['repo'].nunique()} repos"
)
df.head()

## Overall Success Rate by Model

In [ ]:
summary = (
    df.groupby("model")
    .agg(
        total=("success", "count"),
        successes=("success", "sum"),
        success_rate=("success", "mean"),
        mean_cost_usd=("cost_usd", "mean"),
        mean_agent_duration=("agent_duration_seconds", "mean"),
    )
    .sort_values("success_rate", ascending=False)
)

summary["success_rate"] = (summary["success_rate"] * 100).round(1)
summary["mean_cost_usd"] = summary["mean_cost_usd"].round(4)
summary["mean_agent_duration"] = summary["mean_agent_duration"].round(1)
summary

In [ ]:
model_order = summary.index.tolist()  # already sorted best-to-worst

fig = px.bar(
    summary.reset_index(),
    x="model",
    y="success_rate",
    text="success_rate",
    color="model",
    title="Overall Success Rate by Model",
    labels={"success_rate": "Success Rate (%)", "model": "Model"},
    category_orders={"model": model_order},
)
fig.update_traces(texttemplate="%{text}%", textposition="outside")
fig.update_layout(yaxis_range=[0, 105], showlegend=False)
fig.show()

## Success Rate by Difficulty

In [ ]:
diff_df = df.groupby(["difficulty", "model"])["success"].mean().reset_index()
diff_df["success"] = (diff_df["success"] * 100).round(1)

fig = px.bar(
    diff_df,
    x="difficulty",
    y="success",
    color="model",
    barmode="group",
    text="success",
    title="Success Rate by Difficulty & Model",
    labels={"success": "Success Rate (%)", "difficulty": "Difficulty"},
    category_orders={"model": model_order},
)
fig.update_traces(texttemplate="%{text}%", textposition="outside")
fig.update_layout(yaxis_range=[0, 105])
fig.show()

## Success Rate by Language

In [ ]:
lang_df = df.groupby(["language", "model"])["success"].mean().reset_index()
lang_df["success"] = (lang_df["success"] * 100).round(1)

fig = px.bar(
    lang_df,
    x="language",
    y="success",
    color="model",
    barmode="group",
    text="success",
    title="Success Rate by Language & Model",
    labels={"success": "Success Rate (%)", "language": "Language"},
    category_orders={"model": model_order},
)
fig.update_traces(texttemplate="%{text}%", textposition="outside")
fig.update_layout(yaxis_range=[0, 105])
fig.show()

## Per-Repo Comparison (Heatmap)

In [ ]:
pivot = df.pivot_table(index="repo", columns="model", values="success", aggfunc="mean")
pivot = pivot.sort_index()

fig = px.imshow(
    pivot,
    color_continuous_scale="RdYlGn",
    zmin=0,
    zmax=1,
    title="Success by Repo x Model",
    labels={"color": "Success Rate"},
    aspect="auto",
    height=max(400, len(pivot) * 22),
)
fig.show()

## Cost & Duration

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=("Cost vs Success", "Duration vs Success"))

for model in df["model"].unique():
    grp = df[df["model"] == model]
    jitter = np.random.uniform(-0.05, 0.05, len(grp))
    fig.add_trace(
        go.Scatter(
            x=grp["cost_usd"],
            y=grp["success"].astype(int) + jitter,
            mode="markers",
            name=model,
            opacity=0.6,
            marker={"size": 7},
            text=grp["repo"],
            hovertemplate="%{text}<br>Cost: $%{x:.4f}<br>Success: %{y:.0f}",
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=grp["agent_duration_seconds"] / 60,
            y=grp["success"].astype(int) + jitter,
            mode="markers",
            name=model,
            opacity=0.6,
            showlegend=False,
            marker={"size": 7},
            text=grp["repo"],
            hovertemplate="%{text}<br>Duration: %{x:.1f} min<br>Success: %{y:.0f}",
        ),
        row=1,
        col=2,
    )

fig.update_xaxes(title_text="Cost (USD)", row=1, col=1)
fig.update_xaxes(title_text="Agent Duration (min)", row=1, col=2)
fig.update_yaxes(title_text="Success (jittered)", row=1, col=1)
fig.update_layout(height=400, width=900)
fig.show()

In [ ]:
cost_summary = (
    df.groupby("model")
    .agg(
        mean_cost=("cost_usd", "mean"),
        median_cost=("cost_usd", "median"),
        mean_duration_min=("agent_duration_seconds", lambda x: (x / 60).mean()),
        median_duration_min=("agent_duration_seconds", lambda x: (x / 60).median()),
        timeouts=("agent_timed_out", "sum"),
    )
    .round(3)
)
cost_summary

## Common Failure Reasons

In [ ]:
failures = df[~df["success"]].copy()


def categorize_error(msg):
    if not msg:
        return "unknown"
    msg_lower = msg.lower()
    if "dockerfile not found" in msg_lower:
        return "Dockerfile not found"
    if "build failed" in msg_lower:
        return "Build failed"
    if "test" in msg_lower and "fail" in msg_lower:
        return "Tests failed"
    if "timeout" in msg_lower or "timed out" in msg_lower:
        return "Timeout"
    return "Other"


failures["error_category"] = failures["error_message"].apply(categorize_error)
err_df = failures.groupby(["model", "error_category"]).size().reset_index(name="count")

fig = px.bar(
    err_df,
    x="model",
    y="count",
    color="error_category",
    title="Failure Reasons by Model",
    labels={"count": "Count", "error_category": "Error Category"},
    category_orders={"model": model_order},
)
fig.show()

## Head-to-Head: Which repos does each model uniquely solve?

In [ ]:
repo_success = df.groupby(["repo", "model"])["success"].any().unstack(fill_value=False)

for model in repo_success.columns:
    others = [c for c in repo_success.columns if c != model]
    unique = repo_success[model] & ~repo_success[others].any(axis=1)
    unique_repos = unique[unique].index.tolist()
    if unique_repos:
        print(f"\n{model} uniquely solves ({len(unique_repos)}): {', '.join(unique_repos)}")
    else:
        print(f"\n{model}: no uniquely solved repos")